# Agent Platform Training Service を使用した大規模トレーニング
**学習目標:**
  1. トレーニング コードを Python パッケージにまとめる方法を学ぶ
  1. Google Cloud の Agent Platform Training Service を介してクラウド インフラストラクチャを使用してモデルをトレーニングする
  1. Docker コンテナを使用してトレーニング パッケージを実行し、トレーニング用 Docker イメージを Docker レジストリに push する方法を学ぶ
  1. TensorBoard を使用してクラウド ジョブをモニタリングする

## はじめに

このノートブックでは、ローカルでのトレーニングからクラウドでのトレーニングへとステップアップします。Google Cloud の [Agent Platform Training Service](https://console.cloud.google.com/agent-platform/overview) を活用します。

Agent Platform Training Service は、サーバーをプロビジョニングまたは保守することなく ML モデルのトレーニングとデプロイを行えるマネージド サービスです。インフラストラクチャはマネージド サービスによって透過的に処理されます。

各学習目標は、[演習用ラボ ノートブック](../labs/1_training_at_scale.ipynb) の __#TODO__ に対応しています。このソリューション ノートブックを確認する前に、まずはそちらのノートブックの完了に挑戦してみてください。

下のセルでプロジェクト名とバケット名を指定します。

In [ ]:
import os
import warnings
from datetime import datetime

import tensorboard
from google import api_core
from google.cloud import aiplatform, bigquery


%load_ext tensorboard
%load_ext google.cloud.bigquery
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

必要に応じて次のセルを変更してください。

In [ ]:
# 必要に応じて以下を変更してください
PROJECT = !gcloud config get-value project  # noqa: E999
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

OUTDIR = f"gs://{BUCKET}/taxifare/data"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION
%env OUTDIR=$OUTDIR

バケットがリージョナルであり、そのリージョンが指定されたリージョンと一致していることを以下で確認します。

In [ ]:
%%bash
gcloud storage ls -L -b gs://$BUCKET | grep "gs://\|Location"
echo $REGION

In [ ]:
%%bash
gcloud config set project $PROJECT
gcloud config set ai/region $REGION

## BigQuery テーブルの作成

データ用の BigQuery データセットをまだ作成していない場合は、次のセルを実行します。

In [ ]:
bq = bigquery.Client(project=PROJECT)
dataset = bigquery.Dataset(bq.dataset("taxifare"))

try:
    bq.create_dataset(dataset)
    print("Dataset created")
except api_core.exceptions.Conflict:
    print("Dataset already exists")

100 万件のサンプルを含むテーブルを作成しましょう。

このクエリは、`WHERE` 句および `ORDER BY` 句でハッシュ関数（`FARM_FINGERPRINT`）を使用して、ランダム性を導入しつつ再現性を確保するというベスト プラクティスを反映しています。

列の順序は、CSV ファイル内の順序とまったく同じであることに注意してください。

In [ ]:
%%bigquery

CREATE OR REPLACE TABLE taxifare.feateng_training_data AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_datetime,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count*1.0 AS passengers,
    'unused' AS key
FROM `nyc-tlc.yellow.trips` as ny_taxi_trips
WHERE ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 1000)) = 1
AND
    trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0
ORDER BY FARM_FINGERPRINT(TO_JSON_STRING(ny_taxi_trips))

検証データセットのサイズをトレーニング データセットの 1/10 にします。

In [ ]:
%%bigquery

CREATE OR REPLACE TABLE taxifare.feateng_valid_data AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_datetime,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count*1.0 AS passengers,
    'unused' AS key
FROM `nyc-tlc.yellow.trips` as ny_taxi_trips
WHERE ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 10000)) = 2
AND
    trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0
ORDER BY FARM_FINGERPRINT(TO_JSON_STRING(ny_taxi_trips))

## テーブルを CSV ファイルとしてエクスポート

In [ ]:
%%bash

if gcloud storage ls $OUTDIR >/dev/null 2>&1; then
    echo "Deleting current contents of $OUTDIR"
    gcloud storage rm -r $OUTDIR
fi

echo "Extracting training data to $OUTDIR"
bq --location=US extract \
   --destination_format CSV  \
   --field_delimiter "," --noprint_header \
   taxifare.feateng_training_data \
   $OUTDIR/taxi-train-*.csv

echo "Extracting validation data to $OUTDIR"
bq --location=US extract \
   --destination_format CSV  \
   --field_delimiter "," --noprint_header \
   taxifare.feateng_valid_data \
   $OUTDIR/taxi-valid-*.csv

gcloud storage ls -l $OUTDIR

トレーニング データセットと検証データセットの両方が Google Cloud Storage に作成されたことを確認します。

In [ ]:
!gcloud storage ls gs://$BUCKET/taxifare/data

In [ ]:
!gcloud storage cat gs://$BUCKET/taxifare/data/taxi-train-000000000000.csv 2>/dev/null | head -2

In [ ]:
!gcloud storage cat gs://$BUCKET/taxifare/data/taxi-valid-000000000000.csv 2>/dev/null | head -2

## コードを Agent Platform Training Service と互換性のある形にする
コードを Agent Platform Training Service と互換性のあるものにするには、次の変更を行う必要があります。

1. データを Google Cloud Storage にアップロードする
2. コードを trainer Python パッケージに移動する
4. `gcloud` を使用してトレーニング ジョブを送信し、Agent Platform 上でトレーニングを実行する

### コードを Python パッケージに移行する

最初に行うべきことは、トレーニング コードのスニペットを通常の Python パッケージに変換することです。

Python パッケージとは、単に 1 つ以上の `.py` ファイルと、そのディレクトリをパッケージとして識別するための `__init__.py` ファイルの集まりです。`__init__.py` には初期化コードが含まれることもありますが、ここでの目的には空のファイルで十分です。

#### パッケージ ディレクトリの作成

パッケージ ディレクトリには 3 つのファイルが含まれます。

In [ ]:
ls ./taxifare/trainer/

#### 既存のコードを model.py に貼り付ける

Python パッケージでは、ノートブックのセルではなく `.py` ファイル内にコードが存在する必要があります。そこで、[こちらのノートブック](../../introduction_to_tensorflow/solutions/5_custom_feature_engineering.ipynb) で開発した既存のコードを 1 つのファイルにコピー＆ペーストします。

In [ ]:
%%writefile ./taxifare/trainer/model.py
"""DNN モデルのデータ準備、トレーニング、評価。"""

import logging
import os

import numpy as np
import tensorflow as tf
import keras
from keras import callbacks
from keras.layers import (
    Concatenate,
    Dense,
    Discretization,
    Embedding,
    Flatten,
    HashedCrossing,
    Input,
    Lambda,
)
from keras.metrics import RootMeanSquaredError

def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = tf.strings.to_number(ds[0])
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    feature = tf.strings.to_number(ds[2:6])  # 一部の特徴量のみを使用
    # 個別に処理できるように特徴量をタプルで渡す
    return (feature[0], feature[1], feature[2], feature[3]), label


def create_dataset(pattern, batch_size, num_repeat, mode="eval"):
    ds = tf.data.Dataset.list_files(pattern)
    ds = ds.flat_map(tf.data.TextLineDataset)
    ds = ds.map(parse_csv)
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.repeat(num_repeat).batch(batch_size, drop_remainder=True)
    return ds

def parse_lat_lon(row):
    columns = tf.strings.split(row, ",")
    # 緯度インデックス: 3 および 5、経度インデックス: 2 および 4
    lat_strings = tf.gather(columns, [3, 5])
    lon_strings = tf.gather(columns, [2, 4])
    lat_features = tf.strings.to_number(lat_strings)
    lon_features = tf.strings.to_number(lon_strings)
    return lat_features, lon_features

def adapt_normalize(train_data_path):
    ds = tf.data.Dataset.list_files(train_data_path)
    ds = ds.flat_map(tf.data.TextLineDataset)
    lat_lon_ds = ds.map(parse_lat_lon).batch(10000)

    lat_scaler = keras.layers.Normalization(axis=None)
    lon_scaler = keras.layers.Normalization(axis=None)

    lat_values, lon_values = next(iter(lat_lon_ds))

    lat_scaler.adapt(lat_values)
    lon_scaler.adapt(lon_values)

    print("Computed statistics for latitude:")
    print(f"mean: {lat_scaler.mean}, variance: {lat_scaler.variance}")
    print("+++++")
    print("Computed statistics for longitude:")
    print(f"mean: {lon_scaler.mean}, variance: {lon_scaler.variance}")

    return lat_scaler, lon_scaler


def euclidean(params):
    lon1, lat1, lon2, lat2 = params
    londiff = lon2 - lon1
    latdiff = lat2 - lat1
    return tf.sqrt(londiff * londiff + latdiff * latdiff)


def transform(inputs, nbuckets, normalizers):
    lat_scaler, lon_scaler = normalizers

    # 経度を正規化
    scaled_plon = lon_scaler(inputs["pickup_longitude"])
    scaled_dlon = lon_scaler(inputs["dropoff_longitude"])

    # 緯度を正規化
    scaled_plat = lat_scaler(inputs["pickup_latitude"])
    scaled_dlat = lat_scaler(inputs["dropoff_latitude"])

    # カスタム euclidean 関数のための Lambda レイヤ
    euclidean_distance = Lambda(euclidean, name="euclidean")(
        [scaled_plon, scaled_plat, scaled_dlon, scaled_dlat]
    )

    # 離散化（バケット化）
    latbuckets = np.linspace(start=-5, stop=5, num=nbuckets).tolist()
    lonbuckets = np.linspace(start=-5, stop=5, num=nbuckets).tolist()

    plon = Discretization(lonbuckets, name="plon_bkt")(scaled_plon)
    plat = Discretization(latbuckets, name="plat_bkt")(scaled_plat)
    dlon = Discretization(lonbuckets, name="dlon_bkt")(scaled_dlon)
    dlat = Discretization(latbuckets, name="dlat_bkt")(scaled_dlat)

    # HashedCrossing レイヤによる特徴量クロス
    p_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 2, name="p_fc")((plon, plat))
    d_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 2, name="d_fc")((dlon, dlat))
    pd_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 4, name="pd_fc")((p_fc, d_fc))

    # Embedding レイヤによる埋め込み
    pd_embed = Flatten()(
        Embedding(input_dim=(nbuckets + 1) ** 4, output_dim=10, name="pd_embed")(
            pd_fc
        )
    )

    transformed = Concatenate()([
        scaled_plon,
        scaled_dlon,
        scaled_plat,
        scaled_dlat,
        euclidean_distance, 
        pd_embed
    ])

    return transformed

def build_dnn_model(nbuckets, nnsize, lr, normalizers):
    INPUT_COLS = [
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
    ]

    inputs = {
        colname: Input(name=colname, shape=(1,), dtype="float32")
        for colname in INPUT_COLS
    }

    # transforms
    x = transform(inputs, nbuckets, normalizers)

    for layer, nodes in enumerate(nnsize):
        x = Dense(nodes, activation="relu", name=f"h{layer}")(x)
    output = Dense(1, name="fare")(x)

    model = keras.Model(inputs=list(inputs.values()), outputs=output)
    # TODO 1a
    lr_optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=lr_optimizer, loss="mse", metrics=[RootMeanSquaredError()])

    return model


def train_and_evaluate(hparams):
    # TODO 1b
    batch_size = hparams["batch_size"]
    nbuckets = hparams["nbuckets"]
    lr = hparams["lr"]
    nnsize = [int(s) for s in hparams["nnsize"].split()]
    eval_data_path = hparams["eval_data_path"]
    num_evals = hparams["num_evals"]
    num_examples_to_train_on = hparams["num_examples_to_train_on"]
    output_dir = hparams["output_dir"]
    train_data_path = hparams["train_data_path"]

    model_export_path = os.path.join(output_dir, "model.keras")
    serving_model_export_path = os.path.join(output_dir, "savedmodel")
    checkpoint_path = os.path.join(output_dir, "checkpoint.keras")
    tensorboard_path = os.path.join(output_dir, "tensorboard")

    if tf.io.gfile.exists(output_dir):
        tf.io.gfile.rmtree(output_dir)

    normalizers = adapt_normalize(eval_data_path)

    model = build_dnn_model(nbuckets, nnsize, lr, normalizers)
    logging.info(model.summary())
    
    trainds = create_dataset(
        pattern=train_data_path, batch_size=batch_size, num_repeat=None, mode="train"
    )

    evalds = create_dataset(
        pattern=eval_data_path, batch_size=batch_size, num_repeat=1, mode="eval"
    )

    steps_per_epoch = num_examples_to_train_on // (batch_size * num_evals)

    checkpoint_cb = callbacks.ModelCheckpoint(
        checkpoint_path, verbose=1
    )
    tensorboard_cb = callbacks.TensorBoard(tensorboard_path, histogram_freq=1)

    history = model.fit(
        trainds,
        validation_data=evalds,
        epochs=num_evals,
        steps_per_epoch=max(1, steps_per_epoch),
        verbose=2,  # 0=silent, 1=progress bar, 2=one line per epoch
        callbacks=[checkpoint_cb, tensorboard_cb],
    )

    # Keras モデル ファイルを保存
    model.save(model_export_path)
    # サービング用に SavedModel 形式でモデルをエクスポート
    model.export(serving_model_export_path)
    return history


### コマンドライン パーサーの定義

上記をよく見ると、モデルを実際にトレーニングするコードをラップする新しい関数 `train_and_evaluate` があることに気づくでしょう。これにより、パラメータの辞書（例: `batch_size`, `num_examples_to_train_on`, `train_data_path` など）をこの関数に渡すことで、トレーニングをパラメータ化できます。

出力ディレクトリ、データ パス、トレーニング ステップ数は、ローカルでトレーニングするかクラウドでトレーニングするかによって異なるため、これは非常に便利です。パラメータ化することで、両方で同じコードを使用できます。

これらのパラメータは、コマンドラインを介して実行時に指定します。つまり、コマンドライン パラメータを解析し、それらのパラメータを使用して `train_and_evaluate()` を呼び出すコードを追加する必要があります。これが `task.py` ファイルの役割です。

In [ ]:
%%writefile taxifare/trainer/task.py
"""`trainer.model` のモデル トレーニング コード用の引数定義。"""

import argparse

from trainer import model

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--batch_size",
        help="トレーニング ステップのバッチサイズ",
        type=int,
        default=32,
    )
    parser.add_argument(
        "--eval_data_path",
        help="評価ファイルの GCS ロケーション パターン",
        required=True,
    )
    parser.add_argument(
        "--nnsize",
        help="隠れ層のサイズ（スペース区切りで指定）",
        default="32 8",
    )
    parser.add_argument(
        "--nbuckets",
        help="緯度と経度を分割するバケット数",
        type=int,
        default=10,
    )
    parser.add_argument(
        "--lr", help="オプティマイザの学習率", type=float, default=0.001
    )
    parser.add_argument(
        "--num_evals",
        help="評価データでモデルを評価する回数",
        type=int,
        default=5,
    )
    parser.add_argument(
        "--num_examples_to_train_on",
        help="トレーニングに使用するサンプル数",
        type=int,
        default=100,
    )
    parser.add_argument(
        "--output_dir",
        help="チェックポイントの書き込みおよびモデルのエクスポート先となる GCS ロケーション",
        required=True,
    )
    parser.add_argument(
        "--train_data_path",
        help="トレーニング ファイルの GCS ロケーション パターン",
        required=True,
    )
    args = parser.parse_args()
    hparams = args.__dict__

    model.train_and_evaluate(hparams)


### trainer モジュール パッケージをローカルで実行する

ローカルのテストデータを使用して、次のようにトレーニング コードをローカルでテストできます。単一のファイルに対して、小さなバッチサイズと 1 回の評価ステップで、非常に小さなトレーニング ジョブを実行します。

In [ ]:
%%bash

EVAL_DATA_PATH=../data/taxi-traffic-valid*
TRAIN_DATA_PATH=../data/taxi-traffic-train*
OUTPUT_DIR=./taxifare-model

test ${OUTPUT_DIR} && rm -rf ${OUTPUT_DIR}
export PYTHONPATH=${PYTHONPATH}:${PWD}/taxifare

python3 -m trainer.task \
--eval_data_path $EVAL_DATA_PATH \
--output_dir $OUTPUT_DIR \
--train_data_path $TRAIN_DATA_PATH \
--batch_size 5 \
--num_examples_to_train_on 100 \
--num_evals 1 \
--nbuckets 10 \
--lr 0.001 \
--nnsize "32 8"

## Agent Platform 上でトレーニング パッケージを実行する


ローカルのスタンドアロン モードでコードが動作したら、Agent Platform を使用してクラウド上で実行できます。

Agent Platform では、次のいずれかの形式でトレーニング コードを Agent Platform に提供できます。

- **事前構築済みコンテナで使用する Python トレーニング アプリケーション**: ML モデルをトレーニングし、Cloud Storage にエクスポートするコードを含む [Python ソース ディストリビューション](https://packaging.python.org/en/latest/overview/#python-source-distributions) を作成します。このトレーニング アプリケーションは、使用予定の事前構築済みコンテナに含まれるすべての依存関係を利用できます。トレーニングに必要なすべての依存関係が Agent Platform の事前構築済みコンテナのいずれかに含まれている場合は、このオプションを使用します。

- **カスタム コンテナ イメージ**: ML モデルをトレーニングし、Cloud Storage にエクスポートするコードを含む Docker コンテナ イメージを作成します。コードに必要な依存関係をコンテナ イメージ内に含めます。


### 方法 1: 事前構築済みコンテナ
まず、事前構築済みコンテナを使用してクラウド トレーニングを実行してみましょう。

そのためには、コードをソース ディストリビューションとしてパッケージ化する必要があります。これには `setuptools` を使用できます。

In [ ]:
%%writefile taxifare/setup.py
"""`setuptools` を使用してソース ディストリビューションを作成する。"""

from setuptools import find_packages, setup

setup(
    name="taxifare_trainer",
    version="0.1",
    packages=find_packages(),
    include_package_data=True,
    description="Taxifare model training application.",
)

In [ ]:
%%bash
cd taxifare
python ./setup.py sdist --formats=gztar
cd ..

パッケージを Cloud Storage バケットに保存します。

In [ ]:
%%bash
gcloud storage cp taxifare/dist/taxifare_trainer-0.1.tar.gz gs://${BUCKET}/taxifare/

#### Python SDK を使用したカスタム ジョブの送信

このソース ディストリビューションをクラウドに送信するには、Agent Platform Python SDK の [CustomJob](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.CustomJob#google_cloud_aiplatform_CustomJob_get) モジュールを使用し、[worker_pool_spec](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.WorkerPoolSpec) の下で Agent Platform Training サービスのパラメータを指定します。これには以下が含まれます。
- [`machine_spec`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.MachineSpec): トレーニングを実行する単一マシンの仕様。ここでは `machine_type` や [アクセラレータ仕様](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.AcceleratorType) を指定できます。
- `python_package_spec`: 事前構築済みコンテナの URL（`executor_image_uri`）、Python パッケージのパス（`package_uris`）、およびトレーニング アプリケーションに渡すカスタム引数（`args`）など、Python パッケージのランタイムに関する仕様をここで定義して提供します。

これはデータセット全体を対象とするため、完了までに時間がかかります。GCP コンソールの Agent Platform Training セクションからジョブを監視できます。

In [ ]:
BATCH_SIZE = 64
NUM_EXAMPLES_TO_TRAIN_ON = 500000
NUM_EVALS = 100
NBUCKETS = 10
LR = 0.001
NNSIZE = "32 8"

base_path = f"gs://{BUCKET}/taxifare"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

args = [
    "--eval_data_path",
    f"{base_path}/data/taxi-valid*",
    "--output_dir",
    f"{base_path}/trained_model_{timestamp}",
    "--train_data_path",
    f"{base_path}/data/taxi-train*",
    "--batch_size",
    f"{BATCH_SIZE}",
    "--num_examples_to_train_on",
    f"{NUM_EXAMPLES_TO_TRAIN_ON}",
    "--num_evals",
    f"{NUM_EVALS}",
    "--nbuckets",
    f"{NBUCKETS}",
    "--lr",
    f"{LR}",
    "--nnsize",
    f"{NNSIZE}",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "python_package_spec": {
            "executor_image_uri": "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest",
            "package_uris": [f"{base_path}/taxifare_trainer-0.1.tar.gz"],
            "python_module": "trainer.task",
            "args": args,
        },
    }
]

training_job = aiplatform.CustomJob(
    display_name=f"taxifare_{timestamp}",
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{base_path}/staging",
)

training_job.submit()

**注:** [`gcloud ai custom-jobs create`](https://cloud.google.com/sdk/gcloud/reference/ai/custom-jobs/create) を使用して、bash コマンドからカスタム トレーニングを実行することもできます。

参考用の同等の bash コマンド:
```bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
OUTDIR=gs://${BUCKET}/taxifare/trained_model_$TIMESTAMP
JOB_NAME=taxifare_$TIMESTAMP
echo ${OUTDIR} ${REGION} ${JOB_NAME}

PYTHON_PACKAGE_URIS=gs://${BUCKET}/taxifare/taxifare_trainer-0.1.tar.gz
MACHINE_TYPE=n1-standard-4
REPLICA_COUNT=1
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest"
PYTHON_MODULE=trainer.task

# モデルとトレーニングのハイパーパラメータ
BATCH_SIZE=64
NUM_EXAMPLES_TO_TRAIN_ON=500000
NUM_EVALS=100
NBUCKETS=10
LR=0.001
NNSIZE="32 8"

# GCS パス
GCS_PROJECT_PATH=gs://$BUCKET/taxifare
DATA_PATH=$GCS_PROJECT_PATH/data
TRAIN_DATA_PATH=$DATA_PATH/taxi-train*
EVAL_DATA_PATH=$DATA_PATH/taxi-valid*

WORKER_POOL_SPEC="machine-type=$MACHINE_TYPE,\
replica-count=$REPLICA_COUNT,\
executor-image-uri=$PYTHON_PACKAGE_EXECUTOR_IMAGE_URI,\
python-module=$PYTHON_MODULE"

ARGS="--eval_data_path=$EVAL_DATA_PATH,\
--output_dir=$OUTDIR,\
--train_data_path=$TRAIN_DATA_PATH,\
--batch_size=$BATCH_SIZE,\
--num_examples_to_train_on=$NUM_EXAMPLES_TO_TRAIN_ON,\
--num_evals=$NUM_EVALS,\
--nbuckets=$NBUCKETS,\
--lr=$LR,\
--nnsize=$NNSIZE"

gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --python-package-uris=$PYTHON_PACKAGE_URIS \
  --worker-pool-spec=$WORKER_POOL_SPEC \
  --args="$ARGS"
```

#### TensorBoard を開く
`model.fit` で `TensorBoard` コールバックを指定したため、トレーニング アプリケーションは TensorBoard ダッシュボード用の中間ログを保存しました。

[Agent Platform Experiments で TensorBoard をホストする](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/experiments/tensorboard-introduction) こともできますが、ここでは単にこのノートブックから開いてみましょう。

ジョブが開始されるまでは、しばらくデータが表示されない場合があります。コンソールでジョブのステータスを確認し、ここに戻って右上の更新ボタンをクリックして TensorBoard を更新してください。

In [ ]:
%tensorboard --logdir {base_path}/trained_model_{timestamp} --port 8000

### 方法 2: カスタム コンテナを使用する

Agent Platform Training はカスタム コンテナでのトレーニングもサポートしており、ユーザーは任意の ML フレームワークやアルゴリズムがプレインストールされた独自の Docker コンテナを持ち込んで Agent Platform Training 上で実行できます。

この最後のセクションでは、カスタマイズされた Docker イメージを使用してクラウド トレーニング ジョブを送信する方法について説明します。

`./taxifare/trainer` パッケージのコンテナ化には 3 つのステップが含まれます。

* `./taxifare` に Dockerfile を作成する
* Docker イメージをビルドする
* GCP プロジェクトの Google Cloud Artifact Registry に push する

`Dockerfile` では以下を指定します。
1. コード内のすべての依存関係が満たされるようにコンテナをプロビジョニングする方法
2. コンテナ内のどこに trainer パッケージをコピーするか
3. コンテナの実行時にどのコマンドを実行するか（`ENTRYPOINT` 行）

In [ ]:
%%writefile ./taxifare/Dockerfile
FROM us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest
# TODO 3

COPY . /code

WORKDIR /code

ENTRYPOINT ["python3", "-m", "trainer.task"]

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "taxifare_training_container"
IMAGE_URI = f"us-docker.pkg.dev/{PROJECT}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}"

os.environ["IMAGE_URI"] = IMAGE_URI

In [ ]:
%%bash 

PROJECT_DIR=$(cd ./taxifare && pwd)
DOCKERFILE=$PROJECT_DIR/Dockerfile

# Artifact Registry 用に docker コマンドを認証
gcloud auth print-access-token | docker login -u oauth2accesstoken --password-stdin https://us-docker.pkg.dev

docker build $PROJECT_DIR -f $DOCKERFILE -t $IMAGE_URI

docker push $IMAGE_URI

#### Python SDK を使用したカスタム ジョブの送信

上記と同様に、Agent Platform Python SDK を使用して `worker_pool_spec` を定義し、クラウド トレーニングを実行しましょう。

定義はほぼ同じですが、ここでは `python_package_spec` の代わりに、カスタム コンテナ イメージのパスを含む `container_spec` を指定している点に注意してください。

In [ ]:
BATCH_SIZE = 64
NUM_EXAMPLES_TO_TRAIN_ON = 500000
NUM_EVALS = 100
NBUCKETS = 10
LR = 0.001
NNSIZE = "32 8"

base_path = f"gs://{BUCKET}/taxifare"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

args = [
    "--eval_data_path",
    f"{base_path}/data/taxi-valid*",
    "--output_dir",
    f"{base_path}/trained_model_{timestamp}",
    "--train_data_path",
    f"{base_path}/data/taxi-train*",
    "--batch_size",
    f"{BATCH_SIZE}",
    "--num_examples_to_train_on",
    f"{NUM_EXAMPLES_TO_TRAIN_ON}",
    "--num_evals",
    f"{NUM_EVALS}",
    "--nbuckets",
    f"{NBUCKETS}",
    "--lr",
    f"{LR}",
    "--nnsize",
    f"{NNSIZE}",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        # ここでは python_package_spec の代わりに container_spec を指定します
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

training_job = aiplatform.CustomJob(
    display_name=f"taxifare_container_{timestamp}",
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{base_path}/staging",
)

training_job.submit()

**注:** 参考までに、コンテナ トレーニングを実行するための同等の bash コマンドを以下に示します。

```bash
# 出力ディレクトリとジョブ ID
OUTDIR=gs://${BUCKET}/taxifare/trained_model_$TIMESTAMP
JOB_NAME=taxifare_container_$TIMESTAMP
echo ${OUTDIR} ${REGION} ${JOB_NAME}

# モデルとトレーニングのハイパーパラメータ
BATCH_SIZE=64
NUM_EXAMPLES_TO_TRAIN_ON=500000
NUM_EVALS=100
NBUCKETS=10
LR=0.001
NNSIZE="32 8"

# トレーニングに使用する Agent Platform マシン
MACHINE_TYPE=n1-standard-4
REPLICA_COUNT=1

# GCS パス
GCS_PROJECT_PATH=gs://$BUCKET/taxifare
DATA_PATH=$GCS_PROJECT_PATH/data
TRAIN_DATA_PATH=$DATA_PATH/taxi-train*
EVAL_DATA_PATH=$DATA_PATH/taxi-valid*

WORKER_POOL_SPEC="machine-type=$MACHINE_TYPE,\
replica-count=$REPLICA_COUNT,\
container-image-uri=$IMAGE_URI"

ARGS="--eval_data_path=$EVAL_DATA_PATH,\
--output_dir=$OUTDIR,\
--train_data_path=$TRAIN_DATA_PATH,\
--batch_size=$BATCH_SIZE,\
--num_examples_to_train_on=$NUM_EXAMPLES_TO_TRAIN_ON,\
--num_evals=$NUM_EVALS,\
--nbuckets=$NBUCKETS,\
--lr=$LR,\
--nnsize=$NNSIZE"

gcloud ai custom-jobs create \
  --region=$REGION \
  --display-name=$JOB_NAME \
  --worker-pool-spec=$WORKER_POOL_SPEC \
  --args="$ARGS"
```

#### TensorBoard を開く

TensorBoard をもう一度確認してみましょう。今回は、上記の別の TensorBoard インスタンスによって `8000` が使用されているため、別のポート番号を使用します。

In [ ]:
%tensorboard --logdir {base_path}/trained_model_{timestamp} --port 8001

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License